# 11 — Passing-Network Vulnerability: Part 1, Baseline Data

### Building the clean player-level and team-match-level dataset for a future disruption analysis

*2018 & 2022 FIFA World Cup · StatsBomb event data*

**Where this fits:** this is the first of a planned 5-part analysis on passing-network
vulnerability — how much a team's structure depends on individual players, tested by
simulating player removal. The full plan is:

1. **Baseline network + player role data** (this notebook)
2. Single-player removal simulation → per-player vulnerability score
3. Targeted vs. random disruption (Monte Carlo)
4. Two-player combination search
5. Case studies, before/after visuals, team robustness ranking

**This notebook does Part 1 only.** No player removal, no simulation, no optimization,
no outcome prediction — just building and validating a trustworthy baseline dataset that
Parts 2-5 will consume. It lives in its own module (`src/network_vulnerability.py`) and
its own notebook so it doesn't touch any existing analysis in this repo.

**Node/edge definition** is unchanged from the rest of the project: node = player,
directed edge = completed pass, weight = pass count. Progressive-pass logic is reused
from `data_loader.extract_passes` (`is_progressive`), not redefined.

In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import network_vulnerability as nv

PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

player_path = PROC_DIR / 'player_network_baseline.csv'
team_path = PROC_DIR / 'team_network_baseline.csv'

if player_path.exists() and team_path.exists():
    player_network_baseline = pd.read_csv(player_path)
    team_network_baseline = pd.read_csv(team_path)
else:
    player_network_baseline, team_network_baseline = nv.build_baseline_tables()
    player_network_baseline.to_csv(player_path, index=False)
    team_network_baseline.to_csv(team_path, index=False)

print('player_network_baseline:', player_network_baseline.shape)
print('team_network_baseline  :', team_network_baseline.shape)

player_network_baseline: (3758, 12)
team_network_baseline  : (256, 7)


## Validation checks

Before this dataset feeds into any disruption simulation, confirm it's internally
consistent: passes are conserved between the raw event data and the graph, recipients
aren't silently missing, and player names don't collide across teams.

In [2]:
nv.run_validation_checks(player_network_baseline, team_network_baseline)

VALIDATION CHECKS

1. Sum of player outgoing weighted degree == total completed passes?
   256 / 256 team-matches match exactly



2. Completed passes with a missing recipient (excluded from the graph): 0

3. Player names mapped to more than one team in the same match: 0

4. Team-match networks generated: 256
5. Player rows generated: 3758


## Output — `player_network_baseline`

One row per match_id + team + player.

In [3]:
print(f"Shape: {player_network_baseline.shape}")
player_network_baseline.head(10)

Shape: (3758, 12)


,match_id,team,player,passes_sent,passes_received,weighted_in_degree,weighted_out_degree,total_weighted_degree,betweenness_centrality,unique_passing_partners,progressive_passes_sent,progressive_passes_received
0,3857254,Denmark,Kasper Dolberg,10,16,16,10,26,0.063187,9,1,13
1,3857254,Denmark,Christian Dannemann Eriksen,66,67,67,66,133,0.087637,14,7,28
2,3857254,Denmark,Andreas Skov Olsen,19,30,30,19,49,0.158242,10,2,20
3,3857254,Denmark,Rasmus Nissen Kristensen,44,39,39,44,83,0.034066,9,15,16
4,3857254,Denmark,Joakim Mæhle,43,40,40,43,83,0.091850,13,10,27
5,3857254,Denmark,Pierre-Emile Højbjerg,68,68,68,68,136,0.026557,13,23,11
6,3857254,Denmark,Joachim Andersen,62,58,58,62,120,0.243956,14,22,1
7,3857254,Denmark,Kasper Schmeichel,22,13,13,22,35,0.056044,9,12,0
8,3857254,Denmark,Andreas Christensen,78,73,73,78,151,0.000000,10,24,3
9,3857254,Denmark,Simon Thorup Kjær,50,45,45,50,95,0.041209,10,19,0


## Output — `team_network_baseline`

One row per match_id + team.

In [4]:
print(f"Shape: {team_network_baseline.shape}")
team_network_baseline.head(10)

Shape: (256, 7)


,match_id,team,num_players,num_edges,network_density,total_completed_passes,unique_passing_connections
0,3857254,Denmark,15,124,0.590476,544,124
1,3857254,Tunisia,16,97,0.404167,315,97
2,3857255,Spain,16,128,0.533333,993,128
3,3857255,Japan,15,89,0.423810,177,89
4,3857256,Switzerland,15,108,0.514286,346,108
5,3857256,Serbia,16,125,0.520833,407,125
6,3857257,Denmark,16,145,0.604167,590,145
7,3857257,Australia,15,92,0.438095,219,92
8,3857258,Brazil,16,125,0.520833,539,125
9,3857258,Serbia,16,135,0.562500,351,135


## Summary statistics

In [5]:
players_per_team_match = player_network_baseline.groupby(['match_id', 'team']).size()
print("Players per team-match:")
print(players_per_team_match.describe())
print()
print("Completed passes per team-match:")
print(team_network_baseline['total_completed_passes'].describe())

Players per team-match:
count    256.000000
mean      14.679688
std        1.065933
min       12.000000
25%       14.000000
50%       14.000000
75%       16.000000
max       17.000000
dtype: float64

Completed passes per team-match:
count     256.000000
mean      416.066406
std       153.610098
min       137.000000
25%       307.750000
50%       406.000000
75%       502.750000
max      1026.000000
Name: total_completed_passes, dtype: float64


In [6]:
print("Passes sent per player (distribution):")
print(player_network_baseline['passes_sent'].describe())
print()
print("Betweenness centrality per player (distribution):")
print(player_network_baseline['betweenness_centrality'].describe())
print()
print("Unique passing partners per player (distribution):")
print(player_network_baseline['unique_passing_partners'].describe())

Passes sent per player (distribution):
count    3758.000000
mean       28.343002
std        23.348392
min         0.000000
25%        11.000000
50%        22.000000
75%        40.750000
max       206.000000
Name: passes_sent, dtype: float64

Betweenness centrality per player (distribution):
count    3758.000000
mean        0.076078
std         0.063790
min         0.000000
25%         0.026617
50%         0.062336
75%         0.108881
max         0.447650
Name: betweenness_centrality, dtype: float64

Unique passing partners per player (distribution):
count    3758.000000
mean        9.242682
std         2.807680
min         1.000000
25%         8.000000
50%        10.000000
75%        11.000000
max        16.000000
Name: unique_passing_partners, dtype: float64


## Next step

Part 1 output is cached to `data/processed/player_network_baseline.csv` and
`data/processed/team_network_baseline.csv`. Part 2 (single-player removal simulation)
will load these directly rather than rebuilding networks from raw events each time.